# JERICHON — AFML 04: Nested Optuna с централизованным движком исполнения

**Торговая симуляция вынесена в `for_ml/backtesting`; confidence-фильтр и baseline comparison удалены.**\n\n
1. `Best trial value` — это score внутренней оптимизации, а не итоговая доходность внешнего OOS-бэктеста. Внешние периоды специально не используются для выбора trial.
2. В каждой trial конфигурации `dollar_threshold` реально перестраиваются Dollar Bars и все 474 признака. Порог выбирается из небольшого дискретного набора, чтобы один и тот же набор признаков строился максимум один раз за запуск и кэшировался.
3. Бэктест закрывает открытую позицию при противоположном уверенном сигнале: сигнал известен на закрытии бара, исполнение — на открытии следующего бара.
4. В аудит добавлены фактические относительные расстояния TP/SL. `tp_mult=1.7` — это множитель target-volatility, а НЕ TP в 0.17%, 0.6% или 1.7%. Реальный процент зависит от target, поэтому его надо проверять по сделкам.
5. Внешние WF-границы фиксируются по исходной временной шкале, чтобы изменение Dollar Bars не двигало сами OOS-периоды.

**Важно:** модель Random Forest фиксирована; Optuna не оптимизирует число деревьев или глубину. Это исследовательский эксперимент на короткой истории, не гарантия прибыльности.


## 1. Подготовка локальных данных

Используется локальная история AlgoPack. В исходной части notebook признаки строятся на исходном `DOLLAR_THRESHOLD`, чтобы проверить доступность pipeline. Затем функция `build_feature_context_for_threshold()` полностью перестраивает Dollar Bars и 474 признака для каждого порога из `DOLLAR_THRESHOLD_CANDIDATES`.

**Почему предыдущий прогон был слишком быстрым:** он строил Dollar Bars и признаки один раз до Optuna, а в trial менял только CUSUM/Triple Barrier параметры. Поэтому `dollar_threshold` в действительности не оптимизировался. В этой версии разные пороги действительно пересобирают данные. Кэш ограничивает повторные перестройки одним разом на каждый кандидат.


Примечание: подавляется только повторяющийся pandas `PerformanceWarning` о фрагментации DataFrame; остальные предупреждения не скрываются. Это предупреждение не означает ошибку расчётов или аварийную нагрузку, но указывает на потенциально менее эффективное построение признаков.


In [6]:
\
# 0. CONFIG + IMPORTS
from pathlib import Path
import os, sys, inspect, warnings, json, math, importlib.util
import numpy as np
import pandas as pd

# The feature pipeline builds many columns incrementally. Hide only this known pandas
# fragmentation warning; unrelated warnings remain visible. This does not change results.
warnings.filterwarnings(
    "ignore",
    message=r"DataFrame is highly fragmented.*",
    category=pd.errors.PerformanceWarning,
)
import matplotlib.pyplot as plt
import mplcyberpunk
plt.style.use("cyberpunk")
import re
from sklearn.impute import SimpleImputer

TICKER = "GAZP"
DOLLAR_THRESHOLD = 10_000_000
MSK = "Europe/Moscow"
CUSUM_THRESHOLD = 0.001
PT_SL = (1.0, 1.0)
VERTICAL_HORIZON = 50
TARGET_WINDOW = 55
EXPECTED_FEATURES = 474

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "for_ml" / "storage" / "history").exists() and (p / "src" / "data" / "builders").exists():
            return p
    raise FileNotFoundError(
        "Не найден корень JERICHON. Открой notebook из папки репозитория "
        "или укажи PROJECT_ROOT вручную."
    )

ROOT = Path(r"C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data")
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))
HISTORY_ROOT = ROOT / "for_ml" / "storage" / "history"

from src.data.builders.dollar_bars import InformationDrivenBars
from src.data.builders.micro_features import MicroFeatures
from src.data.builders.build_features import FeatureBuilder
from src.data.builders import jerichon_feature_pipeline as jfp

print("Ticker       :", TICKER)
print("Data root    :", ROOT)
print("History root :", HISTORY_ROOT)


Ticker       : GAZP
Data root    : C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data
History root : C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\storage\history


## 1. Загрузка локальной истории

Формат путей сохранён **как в `jerichon_mark_63(3).ipynb`**:

- `data/for_ml/storage/history/realtime/{TICKER}/{YYYY-MM-DD}/trades.parquet`
- `data/for_ml/storage/history/supercandles/{TICKER}/{YYYY-MM-DD}/{tradestats,obstats,orderstats}.parquet`

Файлы только читаются. Если какой-то источник отсутствует, notebook покажет это явно.

In [7]:
def available_dates(history_root, ticker):
    dates = set()
    for kind in ("realtime", "supercandles"):
        p = history_root / kind / ticker
        if p.exists():
            dates.update(x.name for x in p.iterdir() if x.is_dir())
    return sorted(d for d in dates if len(d) == 10 and d[4] == "-" and d[7] == "-")

def load_local_history(history_root, ticker):
    keys = ("candles", "trades", "orderbook", "tradestats", "obstats", "orderstats")
    frames = {k: [] for k in keys}
    dates = available_dates(history_root, ticker)

    if not dates:
        raise FileNotFoundError(f"Для {ticker} не найдены сохранённые даты в {history_root}")

    for day in dates:
        for key in ("candles", "trades", "orderbook"):
            p = history_root / "realtime" / ticker / day / f"{key}.parquet"
            if p.exists():
                try:
                    x = pd.read_parquet(p)
                    if not x.empty:
                        x = x.copy()
                        x["file_date"] = day
                        frames[key].append(x)
                except Exception as e:
                    print(f"WARNING: не удалось прочитать {p}: {e}")

        for key in ("tradestats", "obstats", "orderstats"):
            p = history_root / "supercandles" / ticker / day / f"{key}.parquet"
            if p.exists():
                try:
                    x = pd.read_parquet(p)
                    if not x.empty:
                        x = x.copy()
                        x["file_date"] = day
                        frames[key].append(x)
                except Exception as e:
                    print(f"WARNING: не удалось прочитать {p}: {e}")

    out = {k: (pd.concat(v, ignore_index=True) if v else pd.DataFrame()) for k, v in frames.items()}
    print(f"Saved dates discovered: {len(dates)}")
    print(f"Date range            : {dates[0]} → {dates[-1]}")
    print("\\nLocal rows:")
    display(pd.DataFrame({"source": list(out), "rows": [len(out[k]) for k in out]}))
    missing = [s for s in ("trades", "tradestats", "obstats", "orderstats") if out[s].empty]
    if missing:
        raise RuntimeError(f"Не хватает локальных источников: {missing}. API-запрос не будет выполняться.")
    return out, dates

data, SAVED_DATES = load_local_history(HISTORY_ROOT, TICKER)

Saved dates discovered: 19
Date range            : 2026-09-24 → 2026-10-27
\nLocal rows:


,source,rows
0,candles,16931
1,trades,1700177
2,orderbook,380
3,tradestats,3402
4,obstats,3307
5,orderstats,3421


## 2. Построение Dollar Bars и 474 признаков

Используем подготовленный модуль `jerichon_feature_pipeline.py`, а не копируем его инженерный код в notebook. Важно: **не вызываем** публичный `build_feature_dataset()`, если он сам создаёт `AlgoPackProvider`; вместо этого вызываем локальные функции модуля над уже прочитанными DataFrame.

Если локальная версия модуля не содержит ожидаемые helper-функции, выполнение остановится с понятной ошибкой, а не отправит запрос в API.

In [8]:
# Сначала проверяем интерфейс установленного файла — никаких сетевых вызовов здесь нет.
required_helpers = (
    "_trades_for_bars",
    "build_dollar_bars_from_trades",
    "_base",
    "_build_sc_features",
    "_choose_availability",
    "_attach_source",
)
missing_helpers = [name for name in required_helpers if not hasattr(jfp, name)]
if missing_helpers:
    raise RuntimeError(
        "В установленном jerichon_feature_pipeline.py не найдены helper-функции: "
        f"{missing_helpers}. Проверь, что в проекте стоит актуальная версия pipeline. "
        "Этот notebook намеренно не переключается на API."
    )

print("Helper signatures:")
for name in required_helpers:
    print(f"  {name}{inspect.signature(getattr(jfp, name))}")

Helper signatures:
  _trades_for_bars(trades)
  build_dollar_bars_from_trades(trades, threshold=13000000)
  _base(bars)
  _build_sc_features(raw_sources, bars_for_features)
  _choose_availability(df, source_name)
  _attach_source(bars, feature_frame, raw_source, source)


In [9]:
# Dollar Bars: используем тот же денежный объём VALUE / PRICE, что и в reference.
bars = jfp.build_dollar_bars_from_trades(data["trades"], threshold=DOLLAR_THRESHOLD)
if bars is None or bars.empty:
    raise RuntimeError("Dollar Bar builder вернул пустой результат.")

# Унифицируем имя времени для последующих causal joins.
if "bar_ts" not in bars.columns:
    if "close_time" not in bars.columns:
        raise RuntimeError(f"В bars нет bar_ts/close_time. Колонки: {list(bars.columns)}")
    bars["bar_ts"] = pd.to_datetime(bars["close_time"], errors="coerce", utc=True)
else:
    bars["bar_ts"] = pd.to_datetime(bars["bar_ts"], errors="coerce", utc=True)

bars = (bars.dropna(subset=["bar_ts"])
             .sort_values("bar_ts")
             .drop_duplicates("bar_ts", keep="last")
             .reset_index(drop=True))

print("Dollar Bars:", len(bars))
print("Period:", bars["bar_ts"].min(), "→", bars["bar_ts"].max())
display(bars.head(3))

Dollar Bars: 12398
Period: 2026-09-24 03:59:40+00:00 → 2026-10-27 08:50:47+00:00


,open,high,low,close,volume,value,count,open_time,close_time,bar_ts
0,102.70,102.7,102.7,102.70,98230.0,10088221.0,108,2026-09-24 03:59:40+00:00,2026-09-24 03:59:40+00:00,2026-09-24 03:59:40+00:00
1,102.70,102.8,102.7,102.79,103280.0,10612260.6,285,2026-09-24 03:59:40+00:00,2026-09-24 04:00:02+00:00,2026-09-24 04:00:02+00:00
2,102.79,102.8,102.7,102.71,98210.0,10088402.9,89,2026-09-24 04:00:02+00:00,2026-09-24 04:00:21+00:00,2026-09-24 04:00:21+00:00


In [10]:
# Base 73 + Super Candle 401. Работаем через helper-функции pipeline.
base_raw = jfp._base(bars.copy())
if isinstance(base_raw, tuple):
    # Поддержка версий helper, возвращающих (features, metadata).
    base_raw = base_raw[0]
if not isinstance(base_raw, pd.DataFrame):
    raise TypeError(f"_base должен вернуть DataFrame, получено: {type(base_raw)}")

# Приводим base frame к контракту bar_ts + 73 числовых признака.
if "bar_ts" not in base_raw.columns:
    if "close_time" in base_raw.columns:
        base_raw = base_raw.rename(columns={"close_time": "bar_ts"})
    elif isinstance(base_raw.index, pd.DatetimeIndex):
        base_raw = base_raw.reset_index().rename(columns={base_raw.index.name or "index": "bar_ts"})
    elif "ts" in base_raw.columns:
        base_raw = base_raw.rename(columns={"ts": "bar_ts"})
    else:
        raise RuntimeError(f"Не найден временной ключ в _base output. Колонки: {list(base_raw.columns)[:25]}")

base_raw["bar_ts"] = pd.to_datetime(base_raw["bar_ts"], errors="coerce", utc=True)
base_raw = base_raw.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)

# _build_sc_features может возвращать feature_frames либо (feature_frames, groups).
# Адаптируем Dollar Bars к формату, ожидаемому pipeline.
bars_for_sc = bars.copy()

# В pipeline ожидаются ts и CLOSE.
if "ts" not in bars_for_sc.columns:
    if "bar_ts" in bars_for_sc.columns:
        bars_for_sc["ts"] = bars_for_sc["bar_ts"]
    elif "close_time" in bars_for_sc.columns:
        bars_for_sc["ts"] = bars_for_sc["close_time"]
    else:
        raise KeyError("В bars отсутствует временная колонка ts/bar_ts/close_time")

if "CLOSE" not in bars_for_sc.columns:
    if "close" in bars_for_sc.columns:
        bars_for_sc["CLOSE"] = bars_for_sc["close"]
    else:
        raise KeyError("В bars отсутствует цена закрытия CLOSE/close")

bars_for_sc["ts"] = pd.to_datetime(bars_for_sc["ts"], errors="coerce", utc=True)

bars_for_sc = (
    bars_for_sc.dropna(subset=["ts"])
    .sort_values("ts")
    .reset_index(drop=True))

sc_result = jfp._build_sc_features(data, bars_for_sc)

if isinstance(sc_result, tuple):
    feature_frames = sc_result[0]
    feature_groups = sc_result[1] if len(sc_result) > 1 else {}
else:
    feature_frames = sc_result
    feature_groups = {}

if not isinstance(feature_frames, dict):
    raise TypeError("_build_sc_features должен возвращать dict feature frames.")

print("Base output shape:", base_raw.shape)
print("Super Candle feature groups:", list(feature_frames))

Base output shape: (12398, 74)
Super Candle feature groups: ['tradestats', 'orderstats', 'obstats']


In [11]:
# Availability validation and causal joins, source by source.
# Helpers' outputs are inspected to support the current prepared module contract.
validated = {}
availability_diagnostics = []

for source in ("tradestats", "obstats", "orderstats"):
    if data[source].empty:
        raise RuntimeError(f"Локальный источник {source} пуст.")

    result = jfp._choose_availability(data[source].copy(), source)
    if isinstance(result, tuple):
        validated[source] = result[0]
        if len(result) > 1 and isinstance(result[1], pd.DataFrame):
            availability_diagnostics.append(result[1])
    else:
        validated[source] = result

print("Validated sources:")
for source, frame in validated.items():
    print(f"  {source:12s}: {len(frame):,} rows, columns={len(frame.columns)}")

if availability_diagnostics:
    display(pd.concat(availability_diagnostics, ignore_index=True))

Validated sources:
  tradestats  : 3,402 rows, columns=2
  obstats     : 3,307 rows, columns=2
  orderstats  : 3,421 rows, columns=2


In [12]:
# Normalize the bars frame for the pipeline's attach helper.
bar_frame = bars.copy()
if "bar_ts" not in bar_frame.columns:
    raise RuntimeError("bar_ts missing from Dollar Bars.")

# Resolve each raw source to the corresponding feature frame by name.
def resolve_feature_frame(source, frames):
    if source in frames:
        return frames[source]
    matches = [k for k in frames if source.lower() in str(k).lower()]
    if len(matches) == 1:
        return frames[matches[0]]
    raise RuntimeError(f"Не удалось однозначно сопоставить {source} с feature frame. Доступны: {list(frames)}")

causal = bar_frame.copy()
for source in ("tradestats", "obstats", "orderstats"):
    feature_frame = resolve_feature_frame(source, feature_frames)
    
    # ИСПРАВЛЕНО: правильный порядок аргументов + исходные данные вместо validated
    attached = jfp._attach_source(causal, feature_frame, data[source], source)
    
    # Поддержка helper, возвращающего либо изменённый frame, либо None in-place.
    if isinstance(attached, pd.DataFrame):
        causal = attached
    print(f"{source:12s} attached → {causal.shape}")

# Добавляем base features по точному bar_ts.
base_cols = [c for c in base_raw.columns if c != "bar_ts" and pd.api.types.is_numeric_dtype(base_raw[c])]
if len(base_cols) != 73:
    raise RuntimeError(f"Ожидалось 73 base features, найдено {len(base_cols)}.")

base_join = base_raw[["bar_ts"] + base_cols].copy()
base_join = base_join.rename(columns={c: f"base__{c}" for c in base_cols})
base_cols = [c for c in base_join.columns if c.startswith("base__")]

if "bar_ts" not in causal.columns:
    raise RuntimeError(f"После causal join нет bar_ts. Колонки: {list(causal.columns)[:30]}")

causal["bar_ts"] = pd.to_datetime(causal["bar_ts"], errors="coerce", utc=True)
base_join["bar_ts"] = pd.to_datetime(base_join["bar_ts"], errors="coerce", utc=True)
causal = causal.merge(base_join, on="bar_ts", how="left", validate="one_to_one")

# Identify features and provenance without admitting timestamps into X.
time_cols = {"bar_ts", "bar_ts_utc", "ts", "ts_utc", "close_time"}
provenance_cols = [
    c for c in causal.columns
    if c.endswith("__event_ts") or c.endswith("__available_at")
]
sc_prefixes = ("tradestats__", "obstats__", "orderstats__")
sc_feature_cols = [
    c for c in causal.columns
    if c.startswith(sc_prefixes)
    and c not in provenance_cols
    and pd.api.types.is_numeric_dtype(causal[c])
]
# Не включаем сырые OHLCV/служебные колонки из bars: только канонические 73 + 401.
feature_cols = base_cols + sc_feature_cols

if len(feature_cols) != EXPECTED_FEATURES:
    raise RuntimeError(
        f"Feature inventory mismatch: {len(feature_cols)} numeric ML features, expected {EXPECTED_FEATURES}. "
        "Не продолжаем обучение на непроверенной матрице."
    )

if len(feature_cols) != len(set(feature_cols)):
    raise RuntimeError("Есть дублирующиеся имена признаков.")

FINAL_DATASET = causal[["bar_ts"] + feature_cols + provenance_cols].copy()
FINAL_DATASET = FINAL_DATASET.sort_values("bar_ts").reset_index(drop=True)
X_raw = FINAL_DATASET[feature_cols].copy()

print("FINAL_DATASET:", FINAL_DATASET.shape)
print("Rows:", len(FINAL_DATASET))
print("ML features:", len(feature_cols))
print("Provenance columns:", len(provenance_cols))

tradestats   attached → (12398, 106)
obstats      attached → (12398, 244)
orderstats   attached → (12398, 417)
FINAL_DATASET: (12398, 481)
Rows: 12398
ML features: 474
Provenance columns: 6


## 4. Проверка качества признаков

Нулевые значения не удаляем автоматически. Важные показатели:
- `NaN` и `±inf` по каждому признаку;
- доля нулей (диагностика, не критерий удаления);
- константные признаки;
- полностью пустые признаки;
- дублирующиеся строки и timestamps.

Исходный `FINAL_DATASET` не перезаписывается. Для модели создаётся отдельная копия, в которой бесконечности переводятся в `NaN`; далее imputer обучается только на train.

In [13]:
X_num = X_raw.apply(pd.to_numeric, errors="coerce")
arr = X_num.to_numpy(dtype="float64", copy=True)
finite_mask = np.isfinite(arr)

quality = pd.DataFrame(index=X_num.columns)
quality["missing_pct"] = X_num.isna().mean() * 100
quality["positive_inf"] = np.isposinf(arr).sum(axis=0)
quality["negative_inf"] = np.isneginf(arr).sum(axis=0)
quality["zero_pct"] = (X_num.eq(0).mean() * 100)
quality["unique_values"] = X_num.nunique(dropna=True)
quality["all_missing"] = X_num.isna().all()
quality["constant_or_single_value"] = quality["unique_values"] <= 1
quality["non_finite_total"] = quality["positive_inf"] + quality["negative_inf"]

print("Rows:", len(X_num), "| features:", X_num.shape[1])
print("NaN cells:", int(X_num.isna().sum().sum()))
print("+inf cells:", int(np.isposinf(arr).sum()))
print("-inf cells:", int(np.isneginf(arr).sum()))
print("Zero cells (% of all feature cells):", round(float(X_num.eq(0).to_numpy().mean() * 100), 2))
print("Duplicate bar_ts:", int(FINAL_DATASET["bar_ts"].duplicated().sum()))
print("Duplicate feature rows:", int(X_num.duplicated().sum()))

print("\\nWorst missingness:")
display(quality.sort_values(["missing_pct", "non_finite_total"], ascending=False).head(30))

print("\\nFeatures with infinities:")
display(quality.loc[quality["non_finite_total"] > 0].sort_values("non_finite_total", ascending=False))

print("\\nConstant / all-missing features:")
display(quality.loc[quality["constant_or_single_value"] | quality["all_missing"]])

Rows: 12398 | features: 474
NaN cells: 11690
+inf cells: 0
-inf cells: 0
Zero cells (% of all feature cells): 0.75
Duplicate bar_ts: 0
Duplicate feature rows: 0
\nWorst missingness:


,missing_pct,positive_inf,negative_inf,zero_pct,unique_values,all_missing,constant_or_single_value,non_finite_total
tradestats__ts_vwap_b_s_bps_ret,9.687046,0,0,0.411357,2333,False,False,0
tradestats__ts_flow_x_return_ret,5.460558,0,0,0.024197,2613,False,False,0
obstats__ob_vol_imb_bbo_vs_deep_ret,1.500242,0,0,2.193902,1994,False,False,0
obstats__ob_val_imb_bbo_vs_deep_ret,1.427650,0,0,1.645427,1973,False,False,0
orderstats__os_put_vwap_b_s_bps_ret,0.572673,0,0,0.000000,2966,False,False,0
tradestats__ts_trade_imbalance_ret,0.435554,0,0,0.000000,2947,False,False,0
obstats__ob_total_volume_z3,0.314567,0,0,0.000000,2906,False,False,0
obstats__ob_total_volume_ma3,0.314567,0,0,0.000000,2906,False,False,0
obstats__ob_total_volume_z6,0.314567,0,0,0.000000,2906,False,False,0
obstats__ob_total_volume_ma6,0.314567,0,0,0.000000,2906,False,False,0


\nFeatures with infinities:


,missing_pct,positive_inf,negative_inf,zero_pct,unique_values,all_missing,constant_or_single_value,non_finite_total


\nConstant / all-missing features:


,missing_pct,positive_inf,negative_inf,zero_pct,unique_values,all_missing,constant_or_single_value,non_finite_total


In [14]:
# Separate ML copy: keep source dataset unchanged.
X_model = X_num.replace([np.inf, -np.inf], np.nan).copy()

# Remove only features that cannot carry information at all (all missing or constant).
drop_features = quality.index[quality["all_missing"] | quality["constant_or_single_value"]].tolist()
if drop_features:
    print("Removing non-informative columns from model copy:", len(drop_features))
    print(drop_features[:30])
    X_model = X_model.drop(columns=drop_features)

print("X_model shape:", X_model.shape)
print("Remaining NaN cells (to be handled by train-only imputer):", int(X_model.isna().sum().sum()))

# Optional strict stop for pathological data; does not delete rows.
if X_model.shape[1] == 0:
    raise RuntimeError("После удаления пустых/константных признаков не осталось признаков.")

X_model shape: (12398, 474)
Remaining NaN cells (to be handled by train-only imputer): 11690


## 2. Фиксируем признаки и строим labels внутри каждой конфигурации

`FINAL_DATASET` и `X_model` построены на одних и тех же Dollar Bars. На каждом trial заново строятся CUSUM events и Triple Barrier labels; затем признаки присоединяются по timestamp. `TIME` исключается из бинарной цели, как и в предыдущей версии.

Важная методологическая оговорка: бэктест входит на следующем баре, тогда как Triple Barrier рассчитывает уровни от цены события. В симуляции ниже расстояния TP/SL переносятся на фактический entry. Это воспроизводит прежнюю логику, но gap между сигналом и входом остаётся приближением. После первого прогона стоит отдельно сравнить с исполнением, где уровни пересчитываются строго от entry.

In [22]:
# Configuration and imports for the optimization stage
import time, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, brier_score_loss
from src.backtesting.optuna_backtest_adapter import simulate_oos_fold as canonical_simulate_oos_fold
from src.data.labeling.cusum import cusum_filter
from src.data.labeling.triple_barrier import triple_barrier_labeling
import optuna

SEED = 63
INITIAL_CAPITAL = 100_000.0
COMMISSION = 0.00013   # за каждую сторону сделки
SLIPPAGE = 0.0001     # adverse slippage на каждой стороне
DOLLAR_THRESHOLD_FIXED = DOLLAR_THRESHOLD
DOLLAR_THRESHOLD_CANDIDATES = list(range(5_000_000, 15_000_001, 1_000_000))
TARGET_WINDOW_FIXED = TARGET_WINDOW if "TARGET_WINDOW" in globals() else 40

# Небольшой бюджет по умолчанию; сначала проверь корректность на 20–30 trials.
N_TRIALS_PER_OUTER_FOLD = 25  # 75 trials total; feature rebuild is cached per threshold
MIN_TRADES_PER_OPTIMIZATION = 20

# Sizing is delegated to the canonical engine in data/for_ml/backtesting.
# Optuna tunes stop-risk budget; aggregate gross exposure is hard-capped at 3x equity.
RISK_FRACTION_CANDIDATES = [0.0010, 0.0015, 0.0025, 0.0035, 0.0050]
MAX_GROSS_LEVERAGE = 3.0
RF_N_ESTIMATORS = 350
RF_MAX_DEPTH = 8
RF_MIN_SAMPLES_LEAF = 8
RF_MAX_FEATURES = "sqrt"

# Outer validation periods are chronological validation windows, not train intervals.
# Train always contains all earlier history, after purge.
OUTER_FOLDS = [
    ("WF1", 0.50, 0.65),
    ("WF2", 0.65, 0.80),
    ("WF3", 0.80, 1.00),
]

print("Ticker:", TICKER)
print("Dollar threshold fixed for this experiment:", DOLLAR_THRESHOLD_FIXED)
print("RF is fixed; Optuna tunes data/label/trading configuration only.")
print("Optuna version:", optuna.__version__)


Ticker: GAZP
Dollar threshold fixed for this experiment: 10000000
RF is fixed; Optuna tunes data/label/trading configuration only.
Optuna version: 4.6.0


In [23]:
# Build an immutable bar-level feature table. Do not align it to one fixed set of labels.
if "bar_ts" not in FINAL_DATASET.columns:
    raise RuntimeError("FINAL_DATASET must contain bar_ts.")
feature_cols = list(X_model.columns)
feature_table = pd.concat(
    [pd.to_datetime(FINAL_DATASET["bar_ts"], errors="coerce", utc=True).rename("bar_ts").reset_index(drop=True),
     X_model.reset_index(drop=True)],
    axis=1
)
feature_table = (
    feature_table.dropna(subset=["bar_ts"])
    .sort_values("bar_ts")
    .drop_duplicates("bar_ts", keep="last")
    .set_index("bar_ts")
)
feature_table = feature_table[feature_cols].replace([np.inf, -np.inf], np.nan)

bars_bt = bars.copy()
bars_bt["bar_ts"] = pd.to_datetime(bars_bt["bar_ts"], errors="coerce", utc=True)
for low, up in [("open","OPEN"), ("high","HIGH"), ("low","LOW"), ("close","CLOSE")]:
    if low not in bars_bt.columns and up in bars_bt.columns:
        bars_bt[low] = bars_bt[up]
    bars_bt[low] = pd.to_numeric(bars_bt[low], errors="coerce")
bars_bt = (
    bars_bt.dropna(subset=["bar_ts","open","high","low","close"])
    .sort_values("bar_ts")
    .drop_duplicates("bar_ts", keep="last")
    .reset_index(drop=True)
)
bar_ts_index = pd.DatetimeIndex(bars_bt["bar_ts"])
if not feature_table.index.is_monotonic_increasing:
    raise RuntimeError("Feature table timestamps are not sorted.")
if feature_table.index.has_duplicates:
    raise RuntimeError("Feature table contains duplicate timestamps.")

print("Bars:", len(bars_bt), "| feature rows:", len(feature_table), "| features:", len(feature_cols))
print("Period:", bar_ts_index.min(), "→", bar_ts_index.max())
display(pd.DataFrame([{
    "ticker": TICKER, "bars": len(bars_bt), "features": len(feature_cols),
    "start": str(bar_ts_index.min()), "end": str(bar_ts_index.max()),
    "dollar_threshold_fixed": DOLLAR_THRESHOLD_FIXED
}]))


# Freeze calendar boundaries from the baseline feature timeline. Changing Dollar Bars
# changes event timestamps/count, but must not move the external OOS calendar periods.
BASE_TIMELINE = pd.DatetimeIndex(feature_table.index).sort_values().unique()


Bars: 12398 | feature rows: 12398 | features: 474
Period: 2026-09-24 03:59:40+00:00 → 2026-10-27 08:50:47+00:00


,ticker,bars,features,start,end,dollar_threshold_fixed
0,GAZP,12398,474,2026-09-24 03:59:40+00:00,2026-10-27 08:50:47+00:00,10000000


In [24]:
def build_feature_context_for_threshold(threshold):
    """Rebuild Dollar Bars and the full causal feature matrix for a given threshold.

    This intentionally reruns the expensive feature pipeline. The caller caches the
    result by threshold, so each candidate threshold is rebuilt at most once per run.
    """
    # Dollar Bars: используем тот же денежный объём VALUE / PRICE, что и в reference.
    bars = jfp.build_dollar_bars_from_trades(data["trades"], threshold=threshold)
    if bars is None or bars.empty:
        raise RuntimeError("Dollar Bar builder вернул пустой результат.")

    # Унифицируем имя времени для последующих causal joins.
    if "bar_ts" not in bars.columns:
        if "close_time" not in bars.columns:
            raise RuntimeError(f"В bars нет bar_ts/close_time. Колонки: {list(bars.columns)}")
        bars["bar_ts"] = pd.to_datetime(bars["close_time"], errors="coerce", utc=True)
    else:
        bars["bar_ts"] = pd.to_datetime(bars["bar_ts"], errors="coerce", utc=True)

    bars = (bars.dropna(subset=["bar_ts"])
                 .sort_values("bar_ts")
                 .drop_duplicates("bar_ts", keep="last")
                 .reset_index(drop=True))


    # Base 73 + Super Candle 401. Работаем через helper-функции pipeline.
    base_raw = jfp._base(bars.copy())
    if isinstance(base_raw, tuple):
        # Поддержка версий helper, возвращающих (features, metadata).
        base_raw = base_raw[0]
    if not isinstance(base_raw, pd.DataFrame):
        raise TypeError(f"_base должен вернуть DataFrame, получено: {type(base_raw)}")

    # Приводим base frame к контракту bar_ts + 73 числовых признака.
    if "bar_ts" not in base_raw.columns:
        if "close_time" in base_raw.columns:
            base_raw = base_raw.rename(columns={"close_time": "bar_ts"})
        elif isinstance(base_raw.index, pd.DatetimeIndex):
            base_raw = base_raw.reset_index().rename(columns={base_raw.index.name or "index": "bar_ts"})
        elif "ts" in base_raw.columns:
            base_raw = base_raw.rename(columns={"ts": "bar_ts"})
        else:
            raise RuntimeError(f"Не найден временной ключ в _base output. Колонки: {list(base_raw.columns)[:25]}")

    base_raw["bar_ts"] = pd.to_datetime(base_raw["bar_ts"], errors="coerce", utc=True)
    base_raw = base_raw.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)

    # _build_sc_features может возвращать feature_frames либо (feature_frames, groups).
    # Адаптируем Dollar Bars к формату, ожидаемому pipeline.
    bars_for_sc = bars.copy()

    # В pipeline ожидаются ts и CLOSE.
    if "ts" not in bars_for_sc.columns:
        if "bar_ts" in bars_for_sc.columns:
            bars_for_sc["ts"] = bars_for_sc["bar_ts"]
        elif "close_time" in bars_for_sc.columns:
            bars_for_sc["ts"] = bars_for_sc["close_time"]
        else:
            raise KeyError("В bars отсутствует временная колонка ts/bar_ts/close_time")

    if "CLOSE" not in bars_for_sc.columns:
        if "close" in bars_for_sc.columns:
            bars_for_sc["CLOSE"] = bars_for_sc["close"]
        else:
            raise KeyError("В bars отсутствует цена закрытия CLOSE/close")

    bars_for_sc["ts"] = pd.to_datetime(bars_for_sc["ts"], errors="coerce", utc=True)

    bars_for_sc = (
        bars_for_sc.dropna(subset=["ts"])
        .sort_values("ts")
        .reset_index(drop=True))

    sc_result = jfp._build_sc_features(data, bars_for_sc)

    if isinstance(sc_result, tuple):
        feature_frames = sc_result[0]
        feature_groups = sc_result[1] if len(sc_result) > 1 else {}
    else:
        feature_frames = sc_result
        feature_groups = {}

    if not isinstance(feature_frames, dict):
        raise TypeError("_build_sc_features должен возвращать dict feature frames.")


    # Availability validation and causal joins, source by source.
    # Helpers' outputs are inspected to support the current prepared module contract.
    validated = {}
    availability_diagnostics = []

    for source in ("tradestats", "obstats", "orderstats"):
        if data[source].empty:
            raise RuntimeError(f"Локальный источник {source} пуст.")

        result = jfp._choose_availability(data[source].copy(), source)
        if isinstance(result, tuple):
            validated[source] = result[0]
            if len(result) > 1 and isinstance(result[1], pd.DataFrame):
                availability_diagnostics.append(result[1])
        else:
            validated[source] = result

    # Normalize the bars frame for the pipeline's attach helper.
    bar_frame = bars.copy()
    if "bar_ts" not in bar_frame.columns:
        raise RuntimeError("bar_ts missing from Dollar Bars.")

    # Resolve each raw source to the corresponding feature frame by name.
    def resolve_feature_frame(source, frames):
        if source in frames:
            return frames[source]
        matches = [k for k in frames if source.lower() in str(k).lower()]
        if len(matches) == 1:
            return frames[matches[0]]
        raise RuntimeError(f"Не удалось однозначно сопоставить {source} с feature frame. Доступны: {list(frames)}")

    causal = bar_frame.copy()
    for source in ("tradestats", "obstats", "orderstats"):
        feature_frame = resolve_feature_frame(source, feature_frames)
    
        # ИСПРАВЛЕНО: правильный порядок аргументов + исходные данные вместо validated
        attached = jfp._attach_source(causal, feature_frame, data[source], source)
    
        # Поддержка helper, возвращающего либо изменённый frame, либо None in-place.
        if isinstance(attached, pd.DataFrame):
            causal = attached

    # Добавляем base features по точному bar_ts.
    base_cols = [c for c in base_raw.columns if c != "bar_ts" and pd.api.types.is_numeric_dtype(base_raw[c])]
    if len(base_cols) != 73:
        raise RuntimeError(f"Ожидалось 73 base features, найдено {len(base_cols)}.")

    base_join = base_raw[["bar_ts"] + base_cols].copy()
    base_join = base_join.rename(columns={c: f"base__{c}" for c in base_cols})
    base_cols = [c for c in base_join.columns if c.startswith("base__")]

    if "bar_ts" not in causal.columns:
        raise RuntimeError(f"После causal join нет bar_ts. Колонки: {list(causal.columns)[:30]}")

    causal["bar_ts"] = pd.to_datetime(causal["bar_ts"], errors="coerce", utc=True)
    base_join["bar_ts"] = pd.to_datetime(base_join["bar_ts"], errors="coerce", utc=True)
    causal = causal.merge(base_join, on="bar_ts", how="left", validate="one_to_one")

    # Identify features and provenance without admitting timestamps into X.
    time_cols = {"bar_ts", "bar_ts_utc", "ts", "ts_utc", "close_time"}
    provenance_cols = [
        c for c in causal.columns
        if c.endswith("__event_ts") or c.endswith("__available_at")
    ]
    sc_prefixes = ("tradestats__", "obstats__", "orderstats__")
    sc_feature_cols = [
        c for c in causal.columns
        if c.startswith(sc_prefixes)
        and c not in provenance_cols
        and pd.api.types.is_numeric_dtype(causal[c])
    ]
    # Не включаем сырые OHLCV/служебные колонки из bars: только канонические 73 + 401.
    feature_cols = base_cols + sc_feature_cols

    if len(feature_cols) != EXPECTED_FEATURES:
        raise RuntimeError(
            f"Feature inventory mismatch: {len(feature_cols)} numeric ML features, expected {EXPECTED_FEATURES}. "
            "Не продолжаем обучение на непроверенной матрице."
        )

    if len(feature_cols) != len(set(feature_cols)):
        raise RuntimeError("Есть дублирующиеся имена признаков.")

    FINAL_DATASET = causal[["bar_ts"] + feature_cols + provenance_cols].copy()
    FINAL_DATASET = FINAL_DATASET.sort_values("bar_ts").reset_index(drop=True)
    X_raw = FINAL_DATASET[feature_cols].copy()


    return bars, FINAL_DATASET, feature_cols

FEATURE_CONTEXT_CACHE = {}

def activate_dollar_threshold(threshold):
    """Switch all globals used by labels/backtest to a fully rebuilt threshold context."""
    global bars_bt, feature_table, bar_ts_index, DOLLAR_THRESHOLD_ACTIVE
    key = int(threshold)
    if key not in FEATURE_CONTEXT_CACHE:
        rebuilt_bars, rebuilt_dataset, rebuilt_cols = build_feature_context_for_threshold(key)
        ds = rebuilt_dataset.copy()
        ds["bar_ts"] = pd.to_datetime(ds["bar_ts"], errors="coerce", utc=True)
        xx = ds[rebuilt_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        ft = pd.concat([ds["bar_ts"].reset_index(drop=True), xx.reset_index(drop=True)], axis=1)
        ft = ft.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last")
        ft = ft.set_index("bar_ts")
        bb = rebuilt_bars.copy()
        if "bar_ts" not in bb.columns:
            bb["bar_ts"] = pd.to_datetime(bb["close_time"], errors="coerce", utc=True)
        else:
            bb["bar_ts"] = pd.to_datetime(bb["bar_ts"], errors="coerce", utc=True)
        for low, up in [("open","OPEN"), ("high","HIGH"), ("low","LOW"), ("close","CLOSE")]:
            if low not in bb.columns and up in bb.columns:
                bb[low] = bb[up]
            bb[low] = pd.to_numeric(bb[low], errors="coerce")
        bb = bb.dropna(subset=["bar_ts","open","high","low","close"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)
        FEATURE_CONTEXT_CACHE[key] = (bb, ft)
    bars_bt, feature_table = FEATURE_CONTEXT_CACHE[key]
    bar_ts_index = pd.DatetimeIndex(bars_bt["bar_ts"])
    DOLLAR_THRESHOLD_ACTIVE = key
    return len(bars_bt), len(feature_table)


In [25]:
def build_trial_dataset(cusum_threshold, tp_mult, sl_mult, vertical_horizon, target_window):
    """Rebuild CUSUM + Triple Barrier labels for one trial; features remain bar-aligned."""
    b = bars_bt.set_index("bar_ts", drop=False).copy()
    b.index = pd.DatetimeIndex(b.index)
    b.index.name = "bar_ts"
    close = b["close"].astype(float)
    events = pd.DatetimeIndex(cusum_filter(close, threshold=float(cusum_threshold)))
    events = events[events.isin(b.index)].drop_duplicates().sort_values()
    if len(events) < 100:
        return None

    label_bars = b.rename(columns={
        "open":"OPEN", "high":"HIGH", "low":"LOW", "close":"CLOSE"
    })
    label_bars = label_bars[["OPEN","HIGH","LOW","CLOSE"]].copy()
    label_bars.index = pd.DatetimeIndex(label_bars.index)
    labels = triple_barrier_labeling(
        bars=label_bars,
        events=events,
        pt_sl=(float(tp_mult), float(sl_mult)),
        vertical_barrier=int(vertical_horizon),
        target_window=int(target_window),
        target_method="ewm",
    )
    if labels is None or labels.empty or not {"label","t1"}.issubset(labels.columns):
        return None
    labels = labels.copy()
    labels.index = pd.to_datetime(labels.index, errors="coerce", utc=True)
    labels = labels[~labels.index.isna()].sort_index()
    labels = labels[labels["label"].isin([-1,0,1])].copy()
    labels["label"] = labels["label"].astype(int)
    labels["t1"] = pd.to_datetime(labels["t1"], errors="coerce", utc=True)
    common = feature_table.index.intersection(labels.index)
    labels = labels.loc[common].sort_index()
    X = feature_table.reindex(labels.index).copy()
    mask = labels["label"].isin([-1,1]) & labels["t1"].notna()
    X = X.loc[mask]
    labels = labels.loc[mask].copy()
    y = labels["label"].map({-1:0, 1:1}).astype(int)
    if len(X) < 150 or y.nunique() < 2:
        return None
    return X, y, labels


def make_rf():
    # Fixed model; these are deliberately NOT Optuna parameters.
    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features=RF_MAX_FEATURES,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1,
    )


def purge_train_indices(train_idx, labels, validation_start):
    # Purge events whose label interval reaches or crosses the validation start.
    t1 = labels.reindex(train_idx)["t1"]
    return train_idx[(t1.notna() & (t1 < validation_start)).to_numpy()]


def temporal_folds(index, specs):
    """Specs are validation start/end fractions; training is all earlier history."""
    n = len(index)
    folds = []
    for name, val_start_frac, val_end_frac in specs:
        a = max(1, int(n * val_start_frac))
        b = min(n, int(n * val_end_frac))
        if b <= a:
            continue
        tr0 = index[:a]
        va = index[a:b]
        if len(tr0) < 80 or len(va) < 20:
            continue
        folds.append({"name":name, "train_candidate":tr0, "val_idx":va,
                      "val_start":va[0], "val_end":index[b-1]})
    return folds


def prepare_xy(X_train, y_train, X_valid):
    # Fit imputer only on training data; no validation leakage.
    imp = SimpleImputer(strategy="median")
    a = pd.DataFrame(imp.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
    b = pd.DataFrame(imp.transform(X_valid), index=X_valid.index, columns=X_valid.columns)
    # RF cannot use all-NaN columns; imputer may preserve them as zeros/NaNs depending on sklearn version.
    a = a.replace([np.inf,-np.inf], np.nan).fillna(0.0)
    b = b.replace([np.inf,-np.inf], np.nan).fillna(0.0)
    return a, b


## 3. Исполнение и objective

Все торговые решения и учёт позиций выполняются общим движком `data/for_ml/backtesting/backtest_engine.py` через адаптер `optuna_backtest_adapter.py`. Исследовательский notebook отвечает только за подготовку данных, обучение модели и подбор параметров.

- Сигнал с закрытия бара исполняется на следующем `OPEN`.
- Разрешены несколько одновременно открытых позиций; однонаправленные сигналы могут добавлять позиции.
- Противоположный сигнал закрывает позиции противоположного направления.
- Риск на сделку задаётся `risk_fraction`; совокупная валовая экспозиция ограничена 3x текущего капитала.
- Сигналы не отфильтровываются по confidence: направление определяется только вероятностью относительно 0.5.
- TP/SL для `event_labels` привязаны к расстояниям от цены события и переносятся на фактическую цену входа; при одновременном касании TP и SL движок использует консервативное исполнение стопа.

Optuna оптимизирует параметры построения событий/барьеров и `risk_fraction`. Реализация исполнения не дублируется в этом notebook.


In [26]:
def simulate_oos_fold(fold, probs, labels, vertical_horizon, risk_fraction):
    """Thin notebook adapter; all trade/execution logic lives in for_ml.backtesting."""
    return canonical_simulate_oos_fold(
        fold=fold,
        probs=probs,
        labels=labels,
        bars=bars_bt,
        initial_capital=INITIAL_CAPITAL,
        commission=COMMISSION,
        slippage=SLIPPAGE,
        vertical_horizon=int(vertical_horizon),
        risk_fraction=float(risk_fraction),
        max_gross_leverage=MAX_GROSS_LEVERAGE,
        session_timezone=MSK,
    )


def eval_model_fold(fold, X, y, labels, vertical_horizon, risk_fraction):
    tr_candidate = fold["train_candidate"]
    tr_idx = purge_train_indices(tr_candidate, labels, fold["val_start"])
    va_idx = fold["val_idx"]
    if len(tr_idx) < 80 or len(va_idx) < 20 or y.loc[tr_idx].nunique() < 2 or y.loc[va_idx].nunique() < 2:
        return None
    Xtr, Xva = prepare_xy(X.loc[tr_idx], y.loc[tr_idx], X.loc[va_idx])
    model = make_rf()
    t1 = labels.reindex(tr_idx)["t1"]
    duration = (t1 - pd.Series(tr_idx, index=tr_idx)).dt.total_seconds().clip(lower=1.0)
    weights = 1.0 / duration
    weights = weights / weights.mean()
    model.fit(Xtr, y.loc[tr_idx], sample_weight=weights.to_numpy())
    p = model.predict_proba(Xva)[:, 1]
    probs = pd.Series(p, index=va_idx)
    bt = simulate_oos_fold(fold, probs, labels, vertical_horizon, risk_fraction)
    yv = y.loc[va_idx]
    try:
        auc = roc_auc_score(yv, p) if yv.nunique() == 2 else np.nan
    except Exception:
        auc = np.nan
    bt.update({
        "n_validation": len(va_idx),
        "balanced_accuracy": balanced_accuracy_score(yv, (p >= 0.5).astype(int)),
        "roc_auc": auc,
        "brier": brier_score_loss(yv, p),
    })
    return bt


## 4. Nested walk-forward optimization

Каждый trial выбирает порог Dollar Bars из заданного диапазона, параметры CUSUM/Triple Barrier и `risk_fraction`. Для каждого внешнего временного фолда Optuna подбирает конфигурацию только на предшествующей истории, затем выбранная конфигурация оценивается на внешнем OOS-периоде.

Торговая симуляция вызывается через общий модуль `for_ml.backtesting.optuna_backtest_adapter`; при изменении правил исполнения достаточно обновить модуль `for_ml.backtesting`, а не копировать новый симулятор в исследовательский notebook.


In [27]:
def inner_specs_for_outer_train(n_events):
    return [("I1", 0.40, 0.55), ("I2", 0.55, 0.70), ("I3", 0.70, 0.85)]


def make_outer_fold_specs():
    timeline = BASE_TIMELINE
    n = len(timeline)
    out = []
    for name, a, b in OUTER_FOLDS:
        train_end_pos = max(1, int(n * a))
        val_end_pos = min(n, int(n * b))
        if val_end_pos <= train_end_pos:
            continue
        out.append({
            "name": name,
            "train_end_ts": timeline[train_end_pos - 1],
            "val_start": timeline[train_end_pos],
            "val_end": timeline[val_end_pos - 1],
        })
    return out


def make_inner_folds(train_index, labels):
    n = len(train_index)
    out = []
    for name, a, b in inner_specs_for_outer_train(n):
        i, j = int(n * a), min(n, int(n * b))
        if i < 80 or j - i < 20:
            continue
        val_end = train_index[j - 1]
        val_idx = train_index[i:j]
        resolved = labels.reindex(val_idx)["t1"] <= val_end
        val_idx = val_idx[resolved.fillna(False).to_numpy()]
        if len(val_idx) < 20:
            continue
        out.append({
            "name": name, "train_candidate": train_index[:i],
            "val_idx": val_idx, "val_start": train_index[i], "val_end": val_end,
        })
    return out


def objective_for_outer_train(trial, outer_train_end_ts):
    dollar_threshold = trial.suggest_categorical("dollar_threshold", DOLLAR_THRESHOLD_CANDIDATES)
    n_bars, n_features = activate_dollar_threshold(dollar_threshold)
    if n_bars < 300 or n_features < 150:
        return -1.0

    cusum_threshold = trial.suggest_float("cusum_threshold", 0.0003, 0.0030, log=True)
    tp_mult = trial.suggest_float("tp_mult", 0.60, 2.00, step=0.10)
    sl_mult = trial.suggest_float("sl_mult", 0.60, 2.00, step=0.10)
    vertical_horizon = trial.suggest_int("vertical_horizon", 10, 100, step=10)
    target_window = trial.suggest_int("target_window", 20, 100, step=10)
    risk_fraction = trial.suggest_categorical("risk_fraction", RISK_FRACTION_CANDIDATES)

    built = build_trial_dataset(cusum_threshold, tp_mult, sl_mult, vertical_horizon, target_window)
    if built is None:
        return -1.0
    X, y, labels = built
    X = X.loc[X.index <= outer_train_end_ts]
    y, labels = y.reindex(X.index), labels.reindex(X.index)
    resolved_train = labels["t1"] < outer_train_end_ts
    mask = resolved_train.fillna(False)
    X, y, labels = X.loc[mask], y.loc[mask], labels.loc[mask]
    if len(X) < 150 or y.nunique() < 2:
        return -1.0

    inner_folds = make_inner_folds(X.index, labels)
    if not inner_folds:
        return -1.0
    fold_returns, fold_trades = [], []
    for f in inner_folds:
        result = eval_model_fold(f, X, y, labels, vertical_horizon, risk_fraction)
        if result is not None:
            fold_returns.append(result["total_return"])
            fold_trades.append(result["trades"])
    if not fold_returns:
        return -1.0

    compounded = float(np.prod(1.0 + np.asarray(fold_returns)) - 1.0)
    total_trades = int(np.sum(fold_trades))
    if total_trades < MIN_TRADES_PER_OPTIMIZATION:
        compounded -= 0.05 * (MIN_TRADES_PER_OPTIMIZATION - total_trades) / MIN_TRADES_PER_OPTIMIZATION
    objective = compounded + 0.20 * float(np.min(fold_returns))
    trial.set_user_attr("dollar_threshold", int(dollar_threshold))
    trial.set_user_attr("inner_fold_returns", [float(x) for x in fold_returns])
    trial.set_user_attr("inner_fold_trades", [int(x) for x in fold_trades])
    trial.set_user_attr("inner_compounded_return", compounded)
    return float(objective)


def run_nested_optuna(n_trials=N_TRIALS_PER_OUTER_FOLD):
    activate_dollar_threshold(DOLLAR_THRESHOLD_FIXED)
    baseline_labels = build_trial_dataset(CUSUM_THRESHOLD, PT_SL[0], PT_SL[1], VERTICAL_HORIZON, TARGET_WINDOW_FIXED)
    if baseline_labels is None:
        raise RuntimeError("Не удалось построить базовые labels для календарных границ.")
    outer_folds = make_outer_fold_specs()
    all_results, best_params_by_fold, all_trials = [], {}, []

    for outer in outer_folds:
        print(f"\\n=== {outer['name']} | train ends={outer['train_end_ts']} | validation={outer['val_start']} → {outer['val_end']} ===")
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=SEED),
            pruner=optuna.pruners.NopPruner(),
        )
        study.optimize(
            lambda trial: objective_for_outer_train(trial, outer["train_end_ts"]),
            n_trials=int(n_trials), show_progress_bar=True, gc_after_trial=True,
        )
        params = dict(study.best_params)
        best_params_by_fold[outer["name"]] = params
        trial_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
        trial_df["outer_fold"] = outer["name"]
        all_trials.append(trial_df)

        activate_dollar_threshold(params["dollar_threshold"])
        built = build_trial_dataset(
            params["cusum_threshold"], params["tp_mult"], params["sl_mult"],
            params["vertical_horizon"], params["target_window"],
        )
        if built is None:
            all_results.append({"fold": outer["name"], **params, "total_return": np.nan, "trades": 0, "status": "NO_DATA"})
            continue
        X, y, labels = built
        tr_idx = X.index[X.index <= outer["train_end_ts"]]
        va_idx = X.index[(X.index >= outer["val_start"]) & (X.index <= outer["val_end"])]
        va_idx = va_idx[(labels.reindex(va_idx)["t1"] <= outer["val_end"]).fillna(False).to_numpy()]
        if len(tr_idx) < 80 or len(va_idx) < 20:
            all_results.append({"fold": outer["name"], **params, "total_return": np.nan, "trades": 0, "status": "TOO_FEW_EVENTS"})
            continue
        fold = {"name": outer["name"], "train_candidate": tr_idx, "val_idx": va_idx,
                "val_start": outer["val_start"], "val_end": outer["val_end"]}
        result = eval_model_fold(fold, X, y, labels, params["vertical_horizon"], params["risk_fraction"])
        if result is None:
            all_results.append({"fold": outer["name"], **params, "total_return": np.nan, "trades": 0, "status": "FOLD_SKIPPED"})
            continue
        all_results.append({
            "fold": outer["name"], **params,
            "total_return": result["total_return"], "trades": result["trades"],
            "win_rate": result["win_rate"], "profit_factor": result["profit_factor"],
            "max_drawdown": result["max_drawdown"], "balanced_accuracy": result["balanced_accuracy"],
            "roc_auc": result["roc_auc"], "brier": result["brier"], "status": "OK",
        })

    return pd.DataFrame(all_results), best_params_by_fold, pd.concat(all_trials, ignore_index=True)


outer_results, best_params_by_fold, optuna_trials = run_nested_optuna()
display(outer_results)
display(pd.DataFrame([{"outer_fold": k, **v} for k, v in best_params_by_fold.items()]))
valid_returns = outer_results["total_return"].dropna().to_numpy(dtype=float)
print("Nested OOS compounded return:",
      float(np.prod(1.0 + valid_returns) - 1.0) if len(valid_returns) else np.nan)


[I 2026-10-27 22:56:07,502] A new study created in memory with name: no-name-0aad65e9-476e-47c2-bb10-4f15f8126f35


\n=== WF1 | train ends=2026-10-13 10:26:51+00:00 | validation=2026-10-13 10:28:34+00:00 → 2026-10-19 07:35:40+00:00 ===


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-10-27 22:57:34,515] Trial 0 finished with value: -0.815534035683883 and parameters: {'dollar_threshold': 5000000, 'cusum_threshold': 0.000849162642709183, 'tp_mult': 1.7000000000000002, 'sl_mult': 1.6, 'vertical_horizon': 90, 'target_window': 60, 'risk_fraction': 0.005}. Best is trial 0 with value: -0.815534035683883.
[I 2026-10-27 22:58:58,929] Trial 1 finished with value: -0.5054570304097079 and parameters: {'dollar_threshold': 14000000, 'cusum_threshold': 0.0006191060920441187, 'tp_mult': 1.3, 'sl_mult': 1.0, 'vertical_horizon': 60, 'target_window': 40, 'risk_fraction': 0.0015}. Best is trial 1 with value: -0.5054570304097079.
[I 2026-10-27 23:00:22,298] Trial 2 finished with value: -0.47872911939620605 and parameters: {'dollar_threshold': 11000000, 'cusum_threshold': 0.0012072676915662838, 'tp_mult': 1.2000000000000002, 'sl_mult': 1.4, 'vertical_horizon': 100, 'target_window': 20, 'risk_fraction': 0.0025}. Best is trial 2 with value: -0.47872911939620605.
[I 2026-10-27 23:0

[I 2026-10-27 23:11:11,554] A new study created in memory with name: no-name-9168f2bd-cb07-4695-8c60-38a6fdfbb8d7


\n=== WF2 | train ends=2026-10-19 07:35:40+00:00 | validation=2026-10-19 07:35:45+00:00 → 2026-10-20 17:21:37+00:00 ===


... (5 outputs left out)

[I 2026-10-27 23:13:33,815] Trial 0 finished with value: -0.9341824236230218 and parameters: {'dollar_threshold': 5000000, 'cusum_threshold': 0.000849162642709183, 'tp_mult': 1.7000000000000002, 'sl_mult': 1.6, 'vertical_horizon': 90, 'target_window': 60, 'risk_fraction': 0.005}. Best is trial 0 with value: -0.9341824236230218.
[I 2026-10-27 23:13:40,671] Trial 1 finished with value: -0.7080259873776931 and parameters: {'dollar_threshold': 14000000, 'cusum_threshold': 0.0006191060920441187, 'tp_mult': 1.3, 'sl_mult': 1.0, 'vertical_horizon': 60, 'target_window': 40, 'risk_fraction': 0.0015}. Best is trial 1 with value: -0.7080259873776931.
[I 2026-10-27 23:13:46,515] Trial 2 finished with value: -0.7215086661971462 and parameters: {'dollar_threshold': 11000000, 'cusum_threshold': 0.0012072676915662838, 'tp_mult': 1.2000000000000002, 'sl_mult': 1.4, 'vertical_horizon': 100, 'target_window': 20, 'risk_fraction': 0.0025}. Best is trial 1 with value: -0.7080259873776931.
[I 2026-10-27 23:1

,fold,dollar_threshold,cusum_threshold,tp_mult,sl_mult,vertical_horizon,target_window,risk_fraction,total_return,trades,win_rate,profit_factor,max_drawdown,balanced_accuracy,roc_auc,brier,status
0,WF1,10000000,0.002968,2.0,2.0,30,100,0.001,-0.034705,112,0.473214,0.522640,0.035584,0.526786,0.539541,0.249544,OK
1,WF2,10000000,0.002968,2.0,2.0,30,100,0.001,-0.026399,116,0.517241,0.622135,0.031158,0.577586,0.606718,0.243309,OK
2,WF3,10000000,0.002968,2.0,2.0,30,100,0.001,-0.034194,126,0.507937,0.564867,0.038017,0.549433,0.558890,0.246742,OK


,outer_fold,dollar_threshold,cusum_threshold,tp_mult,sl_mult,vertical_horizon,target_window,risk_fraction
0,WF1,10000000,0.002968,2.0,2.0,30,100,0.001
1,WF2,10000000,0.002968,2.0,2.0,30,100,0.001
2,WF3,10000000,0.002968,2.0,2.0,30,100,0.001


Nested OOS compounded return: -0.09232333322427577


In [28]:
# Re-evaluate each selected configuration on its untouched outer OOS fold.
oos_prediction_rows, oos_trade_rows = [], []
for outer in make_outer_fold_specs():
    params = best_params_by_fold.get(outer["name"])
    if not params:
        continue
    activate_dollar_threshold(params["dollar_threshold"])
    built = build_trial_dataset(
        params["cusum_threshold"], params["tp_mult"], params["sl_mult"],
        params["vertical_horizon"], params["target_window"],
    )
    if built is None:
        continue
    X, y, labels = built
    tr_idx = X.index[X.index <= outer["train_end_ts"]]
    va_idx = X.index[(X.index >= outer["val_start"]) & (X.index <= outer["val_end"])]
    va_idx = va_idx[(labels.reindex(va_idx)["t1"] <= outer["val_end"]).fillna(False).to_numpy()]
    tr_idx = purge_train_indices(tr_idx, labels, outer["val_start"]) if len(va_idx) else tr_idx
    if len(tr_idx) < 80 or len(va_idx) < 20 or y.loc[tr_idx].nunique() < 2:
        continue
    Xtr, Xva = prepare_xy(X.loc[tr_idx], y.loc[tr_idx], X.loc[va_idx])
    model = make_rf()
    t1 = labels.reindex(tr_idx)["t1"]
    duration = (t1 - pd.Series(tr_idx, index=tr_idx)).dt.total_seconds().clip(lower=1.0)
    weights = 1.0 / duration
    weights = weights / weights.mean()
    model.fit(Xtr, y.loc[tr_idx], sample_weight=weights.to_numpy())
    p = pd.Series(model.predict_proba(Xva)[:, 1], index=va_idx)
    pred_side = (p >= 0.5).astype(int)
    for ts in va_idx:
        oos_prediction_rows.append({
            "fold": outer["name"], "timestamp": ts,
            "y_true": int(y.loc[ts]), "p_up": float(p.loc[ts]),
            "predicted_side": int(pred_side.loc[ts]),
            "correct": int(pred_side.loc[ts] == y.loc[ts]),
        })
    fold = {"name": outer["name"], "train_candidate": tr_idx, "val_idx": va_idx,
            "val_start": outer["val_start"], "val_end": outer["val_end"]}
    bt = simulate_oos_fold(fold, p, labels, params["vertical_horizon"], params["risk_fraction"])
    for r in bt["trade_rows"]:
        r["fold"] = outer["name"]
        r["risk_fraction"] = params["risk_fraction"]
        r["dollar_threshold"] = params["dollar_threshold"]
        oos_trade_rows.append(r)

oos_predictions = pd.DataFrame(oos_prediction_rows)
oos_trades = pd.DataFrame(oos_trade_rows)
if not oos_predictions.empty:
    print("OOS balanced accuracy:",
          balanced_accuracy_score(oos_predictions["y_true"], oos_predictions["predicted_side"]))
    print("OOS ROC AUC:",
          roc_auc_score(oos_predictions["y_true"], oos_predictions["p_up"])
          if oos_predictions["y_true"].nunique() == 2 else np.nan)
else:
    print("No outer OOS predictions available; inspect outer_results/status.")
display(oos_trades.head(20))


OOS balanced accuracy: 0.5503750997605746
OOS ROC AUC: 0.5639584996009577


,signal_time,entry_time,exit_time,side,side_name,qty,entry,exit,entry_notional,exit_notional,...,tp,sl,gross_pnl,fees,net_pnl,reason,bars_held,fold,risk_fraction,dollar_threshold
0,2026-10-13 10:36:09+00:00,2026-10-13 10:36:24+00:00,2026-10-13 10:38:22+00:00,-1,SHORT,696.191391,103.509648,103.663652,72062.525806,72169.742058,...,103.366009,103.653287,-107.216253,18.750195,-125.966447,stop_loss,3,WF1,0.001,10000000
1,2026-10-13 10:56:35+00:00,2026-10-13 10:59:16+00:00,2026-10-13 11:19:06+00:00,-1,SHORT,706.002827,103.349664,103.501477,72965.154933,73072.335470,...,103.208200,103.491128,-107.180536,18.984874,-126.165410,stop_loss,9,WF1,0.001,10000000
2,2026-10-13 11:27:59+00:00,2026-10-13 11:28:00+00:00,2026-10-13 11:42:22+00:00,-1,SHORT,743.581856,103.129686,103.005840,76685.363369,76593.274063,...,102.995541,103.263831,92.089307,19.926223,72.163084,take_profit,17,WF1,0.001,10000000
3,2026-10-13 11:43:26+00:00,2026-10-13 11:43:35+00:00,2026-10-13 11:57:35+00:00,-1,SHORT,786.637050,102.879711,103.016906,80928.992332,81036.915244,...,102.752816,103.006606,-107.922912,21.055568,-128.978480,stop_loss,13,WF1,0.001,10000000
4,2026-10-13 12:02:29+00:00,2026-10-13 12:02:37+00:00,2026-10-13 12:04:09+00:00,-1,SHORT,783.056888,103.159683,103.042676,80779.900345,80688.277313,...,103.032373,103.286993,91.623032,20.990863,70.632169,take_profit,2,WF1,0.001,10000000
5,2026-10-13 12:46:55+00:00,2026-10-13 12:47:05+00:00,2026-10-13 12:47:11+00:00,1,LONG,769.379030,103.440343,103.559651,79584.830744,79676.623970,...,103.570008,103.310678,91.793226,20.703989,71.089237,take_profit,3,WF1,0.001,10000000
6,2026-10-13 12:47:14+00:00,2026-10-13 12:47:15+00:00,2026-10-13 12:48:58+00:00,1,LONG,758.520289,103.750374,103.608397,78696.763646,78589.071178,...,103.881989,103.618759,-107.692467,20.447159,-128.139626,stop_loss,7,WF1,0.001,10000000
7,2026-10-13 12:53:09+00:00,2026-10-13 12:53:10+00:00,2026-10-13 12:54:38+00:00,1,LONG,775.847151,103.950394,103.811501,80649.617052,80541.857427,...,104.078905,103.821883,-107.759626,20.954892,-128.714517,stop_loss,9,WF1,0.001,10000000
8,2026-10-13 12:57:44+00:00,2026-10-13 12:59:16+00:00,2026-10-13 13:01:20+00:00,-1,SHORT,795.871119,103.659633,103.795127,82499.708078,82607.543926,...,103.534517,103.784749,-107.835848,21.463943,-129.299791,stop_loss,3,WF1,0.001,10000000
9,2026-10-13 13:01:52+00:00,2026-10-13 13:02:05+00:00,2026-10-13 13:03:17+00:00,-1,SHORT,784.689782,103.979601,103.863253,81591.730421,81500.433029,...,103.852867,104.106335,91.297392,21.201981,70.095411,take_profit,4,WF1,0.001,10000000


## 5. Итоговые результаты и экспорт

Выгружаются сводка, параметры Optuna, OOS-прогнозы и OOS-сделки.


In [26]:
if not outer_results.empty:
    valid_returns = outer_results["total_return"].dropna().to_numpy(dtype=float)
    stitched_return = float(np.prod(1.0 + valid_returns) - 1.0) if len(valid_returns) else np.nan
    compact_summary = pd.DataFrame([{
        "ticker": TICKER,
        "dollar_threshold_candidates": str(DOLLAR_THRESHOLD_CANDIDATES),
        "outer_folds_evaluated": int(outer_results["total_return"].notna().sum()),
        "stitched_oos_total_return": stitched_return,
        "mean_fold_return": float(np.mean(valid_returns)) if len(valid_returns) else np.nan,
        "worst_fold_return": float(np.min(valid_returns)) if len(valid_returns) else np.nan,
        "total_oos_trades": int(outer_results["trades"].fillna(0).sum()),
        "mean_fold_drawdown": float(outer_results["max_drawdown"].mean()) if "max_drawdown" in outer_results else np.nan,
        "n_optuna_trials_per_fold": N_TRIALS_PER_OUTER_FOLD,
        "rf_parameters_fixed": f"n_estimators={RF_N_ESTIMATORS}; max_depth={RF_MAX_DEPTH}; min_samples_leaf={RF_MIN_SAMPLES_LEAF}",
        "max_gross_leverage": MAX_GROSS_LEVERAGE,
    }])
    display(compact_summary)
    out_dir = ROOT.parent / "research" / "afml" / "afml_04_optuna" / "results"
    out_dir.mkdir(parents=True, exist_ok=True)
    compact_summary.to_csv(out_dir / "optuna_oos_summary.csv", index=False)
    outer_results.to_csv(out_dir / "optuna_outer_folds.csv", index=False)
    optuna_trials.to_csv(out_dir / "optuna_trials.csv", index=False)
    if not oos_predictions.empty:
        oos_predictions.to_csv(out_dir / "optuna_oos_predictions.csv", index=False)
    if not oos_trades.empty:
        oos_trades.to_csv(out_dir / "optuna_oos_trades.csv", index=False)
    print("Saved outputs to:", out_dir.resolve())
    print("OOS trade rows:", len(oos_trades))


,ticker,dollar_threshold_candidates,outer_folds_evaluated,stitched_oos_total_return,mean_fold_return,worst_fold_return,total_oos_trades,mean_fold_drawdown,n_optuna_trials_per_fold,rf_parameters_fixed
0,SMLT,"[5000000, 6000000, 7000000, 8000000, 9000000, ...",3,-0.002731,-0.00091,-0.002731,7,0.00091,20,n_estimators=350; max_depth=8; min_samples_leaf=8


,side,is_win,trades,win_rate,mean_net_pnl,median_net_pnl,sum_net_pnl,mean_return_on_notional,median_return_on_notional,mean_confidence,mean_tp_distance_pct,mean_sl_distance_pct,mean_holding_minutes
0,LONG,False,5,0.0,-58.401020,-45.638689,-292.005099,-0.002338,-0.001826,0.630145,0.004808,0.002282,4.953333
1,LONG,True,2,1.0,9.447697,9.447697,18.895393,0.000378,0.000378,0.628536,0.001779,0.005095,0.000000


,side,exit_reason,trades,win_rate,mean_net_pnl,median_net_pnl,sum_net_pnl,mean_return_on_notional,median_return_on_notional,mean_confidence,mean_tp_distance_pct,mean_sl_distance_pct,mean_holding_minutes
0,LONG,SL,4,0.000000,-69.635415,-70.322373,-278.541658,-0.002788,-0.002814,0.636625,0.005794,0.001389,6.191667
1,LONG,TP,3,0.666667,1.810651,2.946017,5.431952,0.000072,0.000118,0.620433,0.001473,0.005348,0.000000


,metric,is_win,count,mean,median,std
0,confidence,False,5,0.630145,0.614131,0.030229
1,confidence,True,2,0.628536,0.628536,0.002937
2,p_up,False,5,0.630145,0.614131,0.030229
3,p_up,True,2,0.628536,0.628536,0.002937
4,tp_distance_pct,False,5,0.004808,0.004299,0.002671
5,tp_distance_pct,True,2,0.001779,0.001779,0.000368
6,sl_distance_pct,False,5,0.002282,0.002405,0.002247
7,sl_distance_pct,True,2,0.005095,0.005095,0.000615
8,holding_minutes,False,5,4.953333,0.000000,6.905406
9,holding_minutes,True,2,0.000000,0.000000,0.000000


Saved compact outputs to: C:\Users\Сергей Кравцов\afml_optuna_outputs
Trade audit rows in memory: 7
